In [23]:
import pandas as pd

df = pd.read_csv('train.csv')
raw = df.copy()      # copy kept for the before/after comparison
log = []             # data quality log

print('Rows and columns:', df.shape)
df.head()

Rows and columns: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [24]:
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print('Total missing values:', nulls.sum())

log.append(f"Found {nulls.sum()} missing values: " + ", ".join(f"{c} ({n})" for c, n in nulls[nulls > 0].items()) + ".")

Age         177
Cabin       687
Embarked      2
dtype: int64
Total missing values: 866


In [25]:
n_ws = (df['Name'] != df['Name'].str.strip()).sum()
print('Names with extra whitespace:', n_ws)
print('Sex before:', list(df['Sex'].unique()))
print('Embarked before:', list(df['Embarked'].unique()))

df['Name'] = df['Name'].str.strip()
df['Sex'] = df['Sex'].str.strip().str.title()
df['Embarked'] = df['Embarked'].replace({'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'})

print('Sex after:', list(df['Sex'].unique()))
print('Embarked after:', list(df['Embarked'].unique()))

log.append(f"Name: stripped extra whitespace from {n_ws} names.")
log.append("Sex: standardised to title case (male/female to Male/Female).")
log.append("Embarked: expanded codes C, Q, S to Cherbourg, Queenstown, Southampton.")

Names with extra whitespace: 2
Sex before: ['male', 'female']
Embarked before: ['S', 'C', 'Q', nan]
Sex after: ['Male', 'Female']
Embarked after: ['Southampton', 'Cherbourg', 'Queenstown', nan]


In [26]:
def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Age: detect only (these are real passengers)
low, high = iqr_bounds(df['Age'])
n_age = ((df['Age'] < low) | (df['Age'] > high)).sum()
print(f'Age bounds: {low:.1f} to {high:.1f} | outliers: {n_age} | oldest: {df["Age"].max():.0f}')
log.append(f"Age: IQR bounds {low:.1f} to {high:.1f}. {n_age} values above the upper bound (oldest is {df['Age'].max():.0f}). Kept, because these are real passengers.")

# Fare: cap at the upper bound
low, high = iqr_bounds(df['Fare'])
n_fare = ((df['Fare'] < low) | (df['Fare'] > high)).sum()
print(f'Fare bounds: {low:.1f} to {high:.1f} | outliers: {n_fare} | max before: {df["Fare"].max():.1f}')
df['Fare'] = df['Fare'].clip(lower=low, upper=high)
print(f'Fare max after capping: {df["Fare"].max():.1f}')
log.append(f"Fare: IQR bounds {low:.1f} to {high:.1f}. Capped {n_fare} values to the upper bound (max was {raw['Fare'].max():.1f}).")

Age bounds: -6.7 to 64.8 | outliers: 11 | oldest: 80
Fare bounds: -26.7 to 65.6 | outliers: 116 | max before: 512.3
Fare max after capping: 65.6


In [27]:
# Age: median
print('Age mean:', round(df['Age'].mean(), 1), '| median:', df['Age'].median())
n = df['Age'].isnull().sum(); m = df['Age'].median()
df['Age'] = df['Age'].fillna(m)
log.append(f"Age: filled {n} missing with median ({m}); mean was {raw['Age'].mean():.1f}, median is less affected by the older passengers.")

# Embarked: mode
n = df['Embarked'].isnull().sum(); m = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(m)
log.append(f"Embarked: filled {n} missing with mode ({m}).")

# Cabin: forward-fill within the same ticket, then 'Unknown'
n_before = df['Cabin'].isnull().sum()
df['Cabin'] = df.groupby('Ticket')['Cabin'].ffill()
n_ff = n_before - df['Cabin'].isnull().sum()
n_left = df['Cabin'].isnull().sum()
df['Cabin'] = df['Cabin'].fillna('Unknown')
log.append(f"Cabin: forward-filled {n_ff} within the same Ticket (people on one ticket usually share a cabin); the remaining {n_left} set to 'Unknown' ({n_left/len(df)*100:.0f}% of rows, too many to guess).")

print('Cabin forward-filled:', n_ff, '| set to Unknown:', n_left)
print('Missing values left:', df.isnull().sum().sum())

Age mean: 29.7 | median: 28.0
Cabin forward-filled: 5 | set to Unknown: 682
Missing values left: 0


In [28]:
compare = pd.DataFrame({
    'before': [len(raw), raw.isnull().sum().sum(), raw['Age'].isnull().sum(), raw['Cabin'].isnull().sum(), raw['Embarked'].isnull().sum(),
               (raw['Name'] != raw['Name'].str.strip()).sum(), raw['Fare'].max(), raw['Fare'].mean()],
    'after':  [len(df), df.isnull().sum().sum(), df['Age'].isnull().sum(), (df['Cabin'] == 'Unknown').sum(), df['Embarked'].isnull().sum(),
               (df['Name'] != df['Name'].str.strip()).sum(), df['Fare'].max(), df['Fare'].mean()]
}, index=['rows', 'missing values', 'Age missing', 'Cabin missing / Unknown', 'Embarked missing',
          'Names with extra whitespace', 'max Fare', 'mean Fare']).round(2)

compare

,before,after
rows,891.00,891.00
missing values,866.00,0.00
Age missing,177.00,0.00
Cabin missing / Unknown,687.00,682.00
Embarked missing,2.00,0.00
Names with extra whitespace,2.00,0.00
max Fare,512.33,65.63
mean Fare,32.20,24.05


In [29]:
df.to_csv('cleaned_dataset.csv', index=False)
print('Saved cleaned_dataset.csv with', len(df), 'rows')

Saved cleaned_dataset.csv with 891 rows


In [30]:
with open('cleaning_log.md', 'w') as f:
    f.write("# Data Quality Log: Titanic dataset\n\n## Cleaning decisions\n")
    for i, line in enumerate(log, 1):
        f.write(f"{i}. {line}\n")
    f.write("\n## Before vs after\n\n| Metric | Before | After |\n|---|---|---|\n")
    for metric, row in compare.iterrows():
        f.write(f"| {metric} | {row['before']:g} | {row['after']:g} |\n")

print(open('cleaning_log.md').read())

# Data Quality Log: Titanic dataset

## Cleaning decisions
1. Found 866 missing values: Age (177), Cabin (687), Embarked (2).
2. Name: stripped extra whitespace from 2 names.
3. Sex: standardised to title case (male/female to Male/Female).
4. Embarked: expanded codes C, Q, S to Cherbourg, Queenstown, Southampton.
5. Age: IQR bounds -6.7 to 64.8. 11 values above the upper bound (oldest is 80). Kept, because these are real passengers.
6. Fare: IQR bounds -26.7 to 65.6. Capped 116 values to the upper bound (max was 512.3).
7. Age: filled 177 missing with median (28.0); mean was 29.7, median is less affected by the older passengers.
8. Embarked: filled 2 missing with mode (Southampton).
9. Cabin: forward-filled 5 within the same Ticket (people on one ticket usually share a cabin); the remaining 682 set to 'Unknown' (77% of rows, too many to guess).

## Before vs after

| Metric | Before | After |
|---|---|---|
| rows | 891 | 891 |
| missing values | 866 | 0 |
| Age missing | 177 | 0 |
| Ca